In [2]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import re

In [ ]:
# Map street abbreviations to full names 
street_replacements = {
    'HY': 'Highway',
    'BL': 'Boulevard',
    'LN': 'Lane',
    'RD': 'Road',
    'TR': 'Trail',
    'DR': 'Drive',
    'PY': 'Parkway',
    'ST': 'Street',
    'WY': 'Way',
    'AV': 'Avenue'
}

def standardize_address(location):
    """
    Clean and standardize location.
    """
    if not isinstance(location, str):
        return ""
    
     # Convert to uppercase and trim spaces
    text = location.upper().strip()
    
    # Remove spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Replace street abbreviations
    for abbr, full_name in street_replacements.items():
        text = re.sub(r'\b' + abbr + r'\b', full_name.upper(), text)
        
    return text


def get_lat_lon(row):
    """
    Get the latitude and longitude for the row.
    """
    address = row['full_address']
    index = row.name  
    try:
        location = geocode(address, timeout=10)
        if location:
            return pd.Series([location.latitude, location.longitude])
        else:
            return pd.Series([None, None])
    except Exception as e:
        return pd.Series([None, None])

In [ ]:
# Load data
crime_data = pd.read_csv('merged_crime_data.csv')
df = crime_data.loc[(crime_data["LAT"] == 0) & (crime_data["LON"] == 0)]

# Standardize LOCATION
df['LOCATION_standardized'] = df['LOCATION'].apply(standardize_address)
# Build full address strings for geocoding
df['full_address'] = df['LOCATION_standardized'] + ", Los Angeles, CA, USA"

# Initialize Nominatim geocoder
geolocator = Nominatim(user_agent="la-crime-project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Apply geocoding function to each row and extract LAT/LON
df[['LAT', 'LON']] = df.apply(get_lat_lon, axis=1)

df = df.drop(columns=['LOCATION_standardized', 'full_address'])

# Save the updated dataset
df.to_csv('crime_data_geoinfo_missing_geocoded.csv', index=False)